# Notebook Setup & Imports

In [1]:
import sys
from pathlib import Path

# Add project root to PYTHONPATH
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

import torch
import pandas as pd
import numpy as np

# Import Project Modules

In [2]:
from src.dataset import load_data, make_data_loader, make_windows
from src.model import StockMLP, StockCNN
from src.train import train
from src.evaluate import evaluate

# Configuration

In [3]:
DATA_PATH = PROJECT_ROOT / "data" / "train.csv"
BATCH_SIZE = 128
WINDOW_SIZE = 30
EPOCHS = 10
USE_CNN = False  # switch between MLP and CNN
NORMALIZE = True

# Load Dataset

In [4]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Dataset not found. Please place train.csv inside data/ directory."
    )

df = load_data(DATA_PATH, frac=0.01, random_state=42)
print(f"Rows loaded: {len(df):,}")
# load_data already samples the rows
df.describe()


Rows loaded: 210,335


,Date,Open,High,Low,Close,Volume
count,210335,210335.000000,210335.000000,210335.000000,210335.000000,2.103350e+05
mean,1973-05-09 05:07:13.040625,0.671716,1.516526,1.479725,1.494121,2.818582e+05
min,1962-01-02 00:00:00,0.000000,0.076070,0.072738,0.074253,0.000000e+00
25%,1972-01-19 12:00:00,0.000000,0.367216,0.358823,0.361584,1.120000e+04
50%,1974-08-22 00:00:00,0.000000,0.804566,0.788031,0.795600,6.000000e+04
75%,1976-05-11 00:00:00,0.688780,1.618610,1.576404,1.590499,2.600000e+05
max,1978-01-06 00:00:00,20.723618,21.665600,19.781635,20.723618,2.182812e+07
std,NaN,1.612587,2.204111,2.152926,2.176856,6.116196e+05


# Creating Sliding Windows

In [5]:
X, y = make_windows(df, window=WINDOW_SIZE)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Positive ratio:", y.mean())


X shape: (209960, 30)
y shape: (209960,)
Positive ratio: 0.4999190321966089


# Train / Validation Split

In [6]:
split_idx = int(len(X) * 0.8)

X_train, X_val = X[:split_idx], X[split_idx:]
y_train, y_val = y[:split_idx], y[split_idx:]

print(f"Total samples: {len(X):,}")
print(f"Train samples: {len(X_train):,}")
print(f"Validation samples: {len(X_val):,}")


Total samples: 209,960
Train samples: 167,968
Validation samples: 41,992


# DataLoaders

In [7]:
from torch.utils.data import TensorDataset, DataLoader

if NORMALIZE:
    from src.features import normalize
    X_train = normalize(X_train)
    X_val = normalize(X_val)

train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32),
)

val_dataset = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.float32),
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

# Initialize Model

In [8]:
if USE_CNN:
    model = StockCNN()
    print("Using CNN model")
else:
    model = StockMLP(input_dim=WINDOW_SIZE)
    print("Using MLP model")

model

Using MLP model


StockMLP(
  (net): Sequential(
    (0): Linear(in_features=30, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)

# Train Model

In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.BCEWithLogitsLoss()
print("hello")
train(
    model=model,
    loader=train_loader,
    epochs=EPOCHS,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
)


hello
Epoch 1/10 - Loss: 0.5515
Epoch 2/10 - Loss: 0.5302
Epoch 3/10 - Loss: 0.5270
Epoch 4/10 - Loss: 0.5247
Epoch 5/10 - Loss: 0.5230
Epoch 6/10 - Loss: 0.5211
Epoch 7/10 - Loss: 0.5192
Epoch 8/10 - Loss: 0.5184
Epoch 9/10 - Loss: 0.5179
Epoch 10/10 - Loss: 0.5174


# Evaluate on Validation Set

In [10]:
model.to(device)

val_accuracy = evaluate(
    model,
    X_val,
    y_val,
    device=device
)

print(f"Validation Accuracy: {val_accuracy:.4f}")


Validation Accuracy: 0.7450


# Save Trained Model

In [11]:
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

model_path = MODELS_DIR / "stock_model_v1.pt"
torch.save(model.state_dict(), model_path)

print(f"Model saved to: {model_path}")

Model saved to: /home/mega/projects/forth-year/NN/stock-trend/models/stock_model_v1.pt


# Quick Sanity Prediction

In [12]:
model.eval()

sample = torch.tensor(X_val[:5], dtype=torch.float32).to(device)
with torch.no_grad():
    logits = model(sample)
    probs = torch.sigmoid(logits)

print("Predicted probabilities:", probs.cpu().numpy())
print("True labels:", y_val[:5])

Predicted probabilities: [0.19268484 0.5727197  0.59112114 0.14118694 0.920578  ]
True labels: [0 0 1 0 1]
